In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Guess HSL bus engine type per route by matching route-level average gCO2/km
(from emissions.txt) to the nearest engine factor from the LCA CSV.

Inputs
------
- /path/to/emissions.txt
    Must contain BUS rows with:
      - a route identifier column (any of: route_id, route, route_short_name, Route, routeid)
      - an average emission intensity column (prefers: avg_co2_per_vehicle_per_km; fuzzy match allowed)
      - optionally distance columns (distance_km, distance, length_km, distance_m, length_m, etc.)
      - a type column to filter buses (Type/type/vehicle_type); fuzzy match "bus"

- /path/to/LCA_....csv
    Must contain columns for bus engines like:
      "Bus - ICE", "Bus - HEV", "Bus - BEV", "Bus - BEV (two packs)", "Bus - FCEV"
    and rows that sum to a total gCO2 per km. If there is no explicit "Total" row,
    the script sums all rows to get a total per engine.

Outputs
-------
- Prints a summary and writes:
  ./route_engine_guess.csv  -> per-route table with:
     route_id, n_rows,
     mean_avg_gCO2_per_km, assigned_engine, assigned_engine_factor_gCO2_per_km,
     abs_diff_g_per_km, rel_diff_pct,
     (optional) sum_km, sum_emissions_g
"""

import pandas as pd
import numpy as np
import os
import re
from typing import List, Optional

# ----------------------- CONFIG -----------------------
EMISSIONS_PATH = "Data_CO2/emissions.txt"        # <- adjust if needed
LCA_PATH       = "Data_CO2/LCA_gCO2_per_pkm_by_transport_mode.csv"         # <- adjust if needed
OUTPUT_PATH     = "Data_CO2/route_engine_guess.csv"


def read_flex_csv(path):
    for enc in ("utf-8", "utf-8-sig", "latin1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, encoding="latin1")

# 1) Read files
em = read_flex_csv(EMISSIONS_PATH)
lca = read_flex_csv(LCA_PATH)

# 2) Keep only BUS rows, skip tram/train/metro
type_col = next((c for c in em.columns if c.lower() == "type"), None)
if type_col:
    tlower = em[type_col].astype(str).str.lower()
    bus_mask = tlower.str.contains("bus", na=False)
    skip_mask = tlower.isin(["tram", "train", "metro"])
    em_bus = em[bus_mask & ~skip_mask].copy()
else:
    # If no Type column exists, we can’t filter safely; proceed with original but warn in a column
    em_bus = em.copy()
    em_bus["filter_note"] = "Type column missing; not filtered"

# 3) LCA: compute engine totals and BEV average
engine_cols = [c for c in lca.columns if isinstance(c, str) and c.strip().lower().startswith("bus -")]
if not engine_cols:
    raise KeyError("No 'Bus - ...' columns found in LCA CSV.")

engine_totals = lca[engine_cols].apply(pd.to_numeric, errors="coerce").sum()

bev_cols = [c for c in engine_cols if c in ["Bus - BEV", "Bus - BEV (two packs)"]]
bev_avg = engine_totals[bev_cols].mean() if bev_cols else float("nan")

# 4) Simple proxy mapping by exact emission value
value_to_engine = {
    1300: "Bus - ICE",
    1100: "Bus - ICE",
    650:  "Bus - HEV",
    550:  "Bus - HEV",
    230:  "Bus - HEV",
    0:    "Bus - BEV",   # will substitute BEV average factor below
}

# Find emission column and apply mapping
em_col = next((c for c in em_bus.columns if c.lower() == "avg_co2_per_vehicle_per_km"), None)
if em_col is None:
    em_col = next((c for c in em_bus.columns
                   if ("co2" in c.lower() and "km" in c.lower() and ("avg" in c.lower() or "mean" in c.lower()))), None)
if em_col is None:
    raise KeyError("Could not find 'avg_co2_per_vehicle_per_km' (or similar) in emissions.txt")

em_bus[em_col] = pd.to_numeric(em_bus[em_col], errors="coerce")

def engine_for_value(v):
    return value_to_engine.get(v)

def engine_factor_for_value(v, eng):
    if eng == "Bus - BEV":
        return float(bev_avg)
    return float(engine_totals.get(eng, float("nan")))

em_bus["assigned_engine_type"] = em_bus[em_col].map(engine_for_value)
em_bus["assigned_engine_factor_gCO2_per_km"] = [
    engine_factor_for_value(v, eng) if eng is not None else None
    for v, eng in zip(em_bus[em_col], em_bus["assigned_engine_type"])
]

# 5) Write fresh buses-only file
em_bus.to_csv(OUTPUT_PATH, index=False)
print("Wrote:", OUTPUT_PATH)

Wrote: Data_CO2/route_engine_guess.csv


In [11]:
em_bus


,route_id,agency_id,route_short_name,Type,avg_co2_per_vehicle_per_km,avg_passenger_count,assigned_engine_type,assigned_engine_factor_gCO2_per_km
23,1016,HSL,16,A-bus,1100,11.0,Bus - ICE,92.0
24,1020,HSL,20,D-bus,0,18.0,Bus - BEV,30.5
25,1021,HSL,21,A-bus,1100,11.0,Bus - ICE,92.0
26,1021N,HSL,21N,A-bus,1100,11.0,Bus - ICE,92.0
27,1022,HSL,22,A-bus,1100,11.0,Bus - ICE,92.0
...,...,...,...,...,...,...,...,...
428,9993K,HSL,993K,A-bus,0,11.0,Bus - BEV,30.5
429,9994,HSL,994,A-bus,0,11.0,Bus - BEV,30.5
430,9994K,HSL,994K,A-bus,0,11.0,Bus - BEV,30.5
431,9995,HSL,995,A-bus,0,11.0,Bus - BEV,30.5
